In [1]:
from IPython.display import HTML

In [2]:
HTML("""

<style>
    .page {
        font-family: "Trebuchet MS", Arial, sans-serif;
        background-color: #454a52;
        color: #d4d6d9;
        padding: 40px;
        min-height: 100vh;
    }

    /* Navigation Bar */
    .navbar {
        position: fixed;
        top: 0;
        left: 0;
        width: 100%;
        background-color: #265e39;
        display: flex;
        justify-content: center;
        gap: 40px;
        padding: 18px 0;
        box-shadow: 0 3px 8px rgba(0,0,0,.3);
        z-index: 1000;
    }

    .navbar a {
        color: #d4d6d9 !important;
        text-decoration: none;
        font-size: 22px;
        font-weight: bold;
        padding: 10px 18px;
        border-radius: 8px;
        transition: background-color .3s ease,
                    color .3s ease,
                    transform .2s ease;
    }

    .navbar a:hover {
        background-color: #3d8c59;
        transform: translateY(-2px);
    }

    .button-container {
        display: flex;       
        gap: 30px;           
        justify-content: center; 
    }

    .button-container button {
        font-size: 20px;
        padding: 40px 50px;  /
        cursor: pointer;    
    }

</style>

<div class="navbar">
    <a class="active" href="http://localhost:8866/voila/render/home.ipynb">Home</a>
    <a href="http://localhost:8866/voila/render/dataset.ipynb">Dataset</a>
    <a href="http://localhost:8866/voila/render/modeling.ipynb">Modeling</a>
    <a href="http://localhost:8866/voila/render/maps.ipynb">Forecast Maps</a>
    <a href="http://localhost:8866/voila/render/summary.ipynb">Summary</a>
    <a href="http://localhost:8866/voila/render/impact.ipynb">Impact</a>
</div>

<div class="page">
    <h1 style="font-size: 80px; text-align: center"><b>Models Used</b></h1>

    <div style="text-align: center">
        <img src="https://truthout.org/app/uploads/2019/12/IMG_0006smaller.jpg" width="800"></img>
    </div>

    <p>""</p>


</div>

""")

In [3]:
import pandas as pd

In [4]:
palmoil_df = pd.read_csv('/Users/janine/Documents/DSSI/Project/hex_deforestation_forecast.csv')

In [7]:
%run "/Users/janine/Documents/DSSI/Project/notebooks/K_MeanClustering.ipynb"


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


SyntaxError: invalid syntax (1227427792.py, line 1)

SyntaxError: invalid syntax (1227427792.py, line 1)

In [ ]:
YEARS = range(2000, 2019)

In [ ]:
# Imports for XGBoost and Light GBM
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

In [ ]:
import plotly.graph_objects as go
N_LAGS = 2  # lags found from PACF plot

# empty lists to keep track of every single year and corresponding RMSE score
results_rf = []
results_gb = []
results_xgb = []
results_lgb = []

for year in YEARS:
  # subset dataframe so it will sort by each corresponding year
  df = all_provinces_avg[all_provinces_avg['year'] <= year].sort_values(['province', 'year'])

  # Create lag matrix
  X, y = make_lag_matrix(df['defor_frac'], N_LAGS)

  # Chronological train/test split
  split = int(len(X) * 0.7)
  X_train, X_test = X[:split], X[split:]
  y_train, y_test = y[:split], y[split:]

  # Random Forest
  rf_model = RandomForestRegressor(n_estimators=195, max_depth=7, random_state=42)
  rf_model.fit(X_train, y_train)
  rf_preds = rf_model.predict(X_test)

  # Gradient Boosting
  gb_model = GradientBoostingRegressor(n_estimators=235, learning_rate=0.4, random_state=42)
  gb_model.fit(X_train, y_train)
  gb_preds = gb_model.predict(X_test)

  # XGBoost
  model_xgb = XGBRegressor(colsample_bytree=0.3, learning_rate=0.3, max_depth=3, n_estimators=50)
  model_xgb.fit(X_train, y_train)
  xgb_preds = model_xgb.predict(X_test)

  # LightGBM
  model_lgb = LGBMRegressor(learning_rate=0.1, max_depth=4, n_estimators=900, num_leaves=10, colsample_bytree=1.0, random_state=42)
  model_lgb.fit(X_train, y_train)
  lgb_preds = model_lgb.predict(X_test)

  # Compute metrics for every model
  metrics_rf = compute_metrics(y_test, rf_preds, label='defor_frac')
  metrics_gb = compute_metrics(y_test, gb_preds, label='defor_frac')
  metrics_xgb = compute_metrics(y_test, xgb_preds, label='defor_frac')
  metrics_lgb = compute_metrics(y_test, lgb_preds, label='defor_frac')

  # Create dataframes to update each year's RMSE/MAE score per year
  results_rf.append({'Year': year, 'RMSE': metrics_rf['RMSE'], 'MAE': metrics_rf['MAE']})
  results_gb.append({'Year': year, 'RMSE': metrics_gb['RMSE'], 'MAE': metrics_gb['MAE']})
  results_xgb.append({'Year': year, 'RMSE': metrics_xgb['RMSE'], 'MAE': metrics_xgb['MAE']})
  results_lgb.append({'Year': year, 'RMSE': metrics_lgb['RMSE'], 'MAE': metrics_lgb['MAE']})

# Convert to DataFrames
results_rf = pd.DataFrame(results_rf).sort_values('Year')
results_gb = pd.DataFrame(results_gb).sort_values('Year')
results_xgb = pd.DataFrame(results_xgb).sort_values('Year')
results_lgb = pd.DataFrame(results_lgb).sort_values('Year')

# Plot
fig = go.Figure()

# Random Forest line
fig.add_trace(go.Scatter(
  x=results_rf['Year'], y=results_rf['RMSE'],
  mode='lines+markers',
  line=dict(color='#480058'),
  marker=dict(symbol='circle'),
  name='Random Forest'
))

# Gradient Boosting line
fig.add_trace(go.Scatter(
  x=results_gb['Year'], y=results_gb['RMSE'],
  mode='lines+markers',
  line=dict(color='#FFA213', dash='dash'),
  marker=dict(symbol='circle'),
  name='Gradient Boosting'
))

# XGBoost Line
fig.add_trace(go.Scatter(
  x=results_xgb['Year'], y=results_xgb['RMSE'],
  mode='lines+markers',
  line=dict(color='#7CAB8D'),
  marker=dict(symbol='circle'),
  name='XGBoost'
))

# Light GBM line
fig.add_trace(go.Scatter(
  x=results_lgb['Year'], y=results_lgb['RMSE'],
  mode='lines+markers',
  line=dict(color='#B72818', dash='dash'),
  marker=dict(symbol='circle'),
  name='LGBM'
))

# Title and layout aprameters
fig.update_layout(
  title='RMSE of Deforestation Fraction By Year',
  xaxis_title='Year',
  yaxis_title='RMSE',
  xaxis=dict(
      tickmode='array',
      tickvals=list(YEARS), 
      tickangle=-45
  ),
  legend=dict(title='Model'),
  width=800,
  height=600,
)

fig.show()